In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import re, csv
import PyPDF2
import yfinance as yf
from datetime import datetime
from fuzzywuzzy import process, fuzz
from nsepython import nse_eq
import json
import shutil

c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\stockprojvenv\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
sector_links = {
    "Auto": "https://www.niftyindices.com/Factsheet/ind_nifty_auto.pdf",
    "Chemicals": "https://niftyindices.com/Factsheet/Factsheet_Nifty_Chemicals.pdf",
    "FMCG": "https://www.niftyindices.com/Factsheet/ind_nifty_FMCG.pdf",
    "Healthcare": "https://www.niftyindices.com/Factsheet/Factsheet_Nifty_Healthcare_Index.pdf",
    "IT": "https://www.niftyindices.com/Factsheet/ind_nifty_it.pdf",
    "Media": "https://www.niftyindices.com/Factsheet/ind_nifty_media.pdf",
    "Metal": "https://www.niftyindices.com/Factsheet/ind_nifty_metal.pdf",
    "Pharma": "https://www.niftyindices.com/Factsheet/ind_nifty_pharma.pdf",
    "PrivateBank": "https://www.niftyindices.com/Factsheet/ind_nifty_private_bank.pdf",
    "PSUBank": "https://www.niftyindices.com/Factsheet/ind_nifty_psu_bank.pdf",
    "OilGas": "https://www.niftyindices.com/Factsheet/Factsheet_nifty_oil_and_gas.pdf"
}


In [4]:
os.makedirs("../data/raw_pdfs", exist_ok=True)

In [5]:
for sector, url in sector_links.items():
    filename = f"../data/raw_pdfs/{sector.replace(' ', '_')}.pdf"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {sector}")
    else:
        print(f"Failed to download {sector} (Status code: {response.status_code})")

Downloaded: Auto
Downloaded: Chemicals
Downloaded: FMCG
Downloaded: Healthcare
Downloaded: IT
Downloaded: Media
Downloaded: Metal
Downloaded: Pharma
Downloaded: PrivateBank
Downloaded: PSUBank
Downloaded: OilGas


In [6]:
top_stocks = {}
NUM_TOP_STOCKS = 10  # <-- change this to however many you want

for sector in sector_links.keys():
    pdf_path = f"../data/raw_pdfs/{sector.replace(' ', '_')}.pdf"
    if not os.path.exists(pdf_path):
        print(f"Skipping {sector}: PDF file not found.")
        continue

    try:
        extracted_stocks = []

        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)

            for page in reader.pages:
                text = page.extract_text()
                if not text:
                    continue

                # Normalize the text
                text = text.replace("\r", "").replace("\t", " ")
                lines = text.split("\n")

                # 1️⃣ Locate "Top Constituents by Weightage"
                start_index = -1
                for i, line in enumerate(lines):
                    if re.search(r"Top\s+constituents\s+by\s+weightage", line, re.IGNORECASE):
                        start_index = i
                        break

                if start_index == -1:
                    continue  # Not on this page

                # 2️⃣ Process lines after that header
                for i in range(start_index + 1, len(lines)):
                    line = lines[i].strip()

                    # Stop at new section or blank
                    if re.search(r"(Sector|Index|As on|Data as on|Source|#)", line, re.IGNORECASE):
                        break

                    # Skip empty or garbage lines
                    if not line or re.match(r"^\d", line):
                        continue

                    # 3️⃣ Extract Company + Weight(%)
                    # Example line: "Sun Pharmaceutical Industries Ltd. 15.23%"
                    match = re.match(r"^([\w&\.\'\s\-\(\)]+?)\s+([\d\.]+)\s*%?$", line)
                    if match:
                        company_name = match.group(1).strip()
                        weight = float(match.group(2))
                        extracted_stocks.append((company_name, weight))

                    # Stop once we have enough
                    if len(extracted_stocks) >= NUM_TOP_STOCKS:
                        break

                if extracted_stocks:
                    break  # Stop scanning more pages once found

        if extracted_stocks:
            top_stocks[sector] = extracted_stocks
            print(f"{sector}: {extracted_stocks}")
        else:
            print(f"⚠️ Could not extract top constituents for {sector}.")

    except Exception as e:
        print(f"❌ Error extracting {sector}: {e}")

# print("\n✅ Top Stocks Extracted:")
# for sector, stocks in top_stocks.items():
#     print(f"{sector}:")
#     for name, wt in stocks:
#         print(f"   {name} → {wt}%")


Auto: [('Mahindra & Mahindra Ltd.', 24.55), ('Maruti Suzuki India Ltd.', 16.87), ('Bajaj Auto Ltd.', 7.83), ('Eicher Motors Ltd.', 7.66), ('Tata Motors Passenger Vehicles Ltd.', 6.8), ('TVS Motor Company Ltd.', 6.54), ('Hero MotoCorp Ltd.', 5.72), ('Dummy Tata Motors Ltd.', 4.33), ('Samvardhana Motherson International Ltd.', 3.72), ('Ashok Leyland Ltd.', 3.22)]
Chemicals: [('Pidilite Industries Ltd.', 12.39), ('SRF Ltd.', 11.76), ('UPL Ltd.', 11.32), ('Solar Industries India Ltd.', 9.34), ('PI Industries Ltd.', 8.01), ('Coromandel International Ltd.', 6.93), ('Navin Fluorine International Ltd.', 5.78), ('Gujarat Fluorochemicals Ltd.', 4.23), ('Tata Chemicals Ltd.', 3.85), ('Linde India Ltd.', 3.5)]
FMCG: [('ITC Ltd.', 33.98), ('Hindustan Unilever Ltd.', 18.75), ('Nestle India Ltd.', 7.81), ('Tata Consumer Products Ltd.', 6.5), ('Britannia Industries Ltd.', 5.9), ('Varun Beverages Ltd.', 5.45), ('Godrej Consumer Products Ltd.', 3.86), ('United Spirits Ltd.', 3.62), ('Marico Ltd.', 3.27)

In [7]:
os.getcwd()

'c:\\Users\\DELL\\Documents\\PRAXIS\\Projects\\Capstone_Project_final\\backend\\utils'

In [8]:
# --- Load JSON data ---
# ✅ File is in the same folder as your notebook/script (utils)
json_path = os.path.join(os.getcwd(), "top_stocks_code.json")

if not os.path.exists(json_path):
    raise FileNotFoundError(f"❌ JSON file not found at: {json_path}")

with open(json_path, "r", encoding="utf-8") as f:
    top_stocks_codes = json.load(f)
print(f"✅ Loaded top_stocks_codes.json from: {json_path}")

# --- Create CSV ---
# ✅ Save the output one level above utils (../data)
output_dir = os.path.join("..", "data")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "sector_stocks.csv")

if os.path.exists(output_path):
    os.remove(output_path)
    print("🗑️ Existing CSV deleted.")

with open(output_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Sector", "Company Name", "NSE Code", "Weight (%)"])

    for sector, company_data in top_stocks.items():
        for company_name, weight in company_data:
            nse_code = top_stocks_codes.get(sector, {}).get(company_name, "NOT_FOUND")
            writer.writerow([sector, company_name, nse_code, weight])

print(f"✅ Final CSV created successfully at: {output_path}")


✅ Loaded top_stocks_codes.json from: c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\top_stocks_code.json
🗑️ Existing CSV deleted.
✅ Final CSV created successfully at: ..\data\sector_stocks.csv


In [9]:
# DATA_DIR = "data/stocks_data"
# os.makedirs(DATA_DIR, exist_ok=True)

# --- Delete existing folder if it exists ---
DATA_DIR = "../data/stock_data"
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
    print("🗑️ Existing 'stock_data' folder deleted.")
os.makedirs(DATA_DIR, exist_ok=True)

🗑️ Existing 'stock_data' folder deleted.


In [10]:
# --- Paths ---
base_dir = os.getcwd()  # current working directory
data_dir = os.path.join(base_dir, "..", "data")
csv_path = os.path.join(data_dir, "sector_stocks.csv")
output_root = os.path.join(data_dir, "stock_data")

print(f"📁 Working directory: {base_dir}")
print(f"📄 Reading data from: {csv_path}")

📁 Working directory: c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils
📄 Reading data from: c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\data\sector_stocks.csv


In [11]:
# --- Read sector_stocks.csv ---
df = pd.read_csv(csv_path)
print(f"✅ Loaded {len(df)} rows from sector_stocks.csv")

# --- Clean and prepare ---
df = df[df["NSE Code"] != "NOT_FOUND"].copy()  # skip missing tickers
df["Weight (%)"] = pd.to_numeric(df["Weight (%)"], errors="coerce")
df.dropna(subset=["Weight (%)"], inplace=True)

# --- Download period ---
start_date = "2023-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")

# --- Create sector-wise folders and fetch data ---
for sector, group in df.groupby("Sector"):
    print(f"\n📊 Processing sector: {sector}")

    # Sort by weight descending, take top 5
    top_stocks = group.sort_values("Weight (%)", ascending=False).head(5)

    # Create folder for this sector
    sector_dir = os.path.join(output_root, sector)
    os.makedirs(sector_dir, exist_ok=True)

    for _, row in top_stocks.iterrows():
        nse_code = row["NSE Code"]
        ticker = f"{nse_code}.NS"
        output_file = os.path.join(sector_dir, f"{nse_code}.csv")

        print(f"   ⏬ Fetching {ticker} ...", end=" ")

        try:
            data = yf.download(ticker, start=start_date, end=end_date, progress=False)

            # Flatten multi-level columns if any
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)

            if data.empty:
                print("⚠️ No data found.")
                continue

            # Keep only Date and Close
            stock_df = data[["Close"]].reset_index()
            stock_df.to_csv(output_file, index=False)
            print(f"✅ Saved {len(stock_df)} rows -> {output_file}")

        except Exception as e:
            print(f"❌ Error fetching {ticker}: {e}")

print("\n🎯 All sectors processed and saved in 'backend/data/stock_data/'")

✅ Loaded 110 rows from sector_stocks.csv

📊 Processing sector: Auto
   ⏬ Fetching M&M.NS ... ✅ Saved 703 rows -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\data\stock_data\Auto\M&M.csv
   ⏬ Fetching MARUTI.NS ... ✅ Saved 702 rows -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\data\stock_data\Auto\MARUTI.csv
   ⏬ Fetching BAJAJ-AUTO.NS ... ✅ Saved 703 rows -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\data\stock_data\Auto\BAJAJ-AUTO.csv
   ⏬ Fetching EICHERMOT.NS ... ✅ Saved 702 rows -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\data\stock_data\Auto\EICHERMOT.csv
   ⏬ Fetching TATAMOTORS.NS ... ✅ Saved 703 rows -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\data\stock_data\Auto\TATAMOTORS.csv

📊 Processing sector: Chemicals
   ⏬ Fetching PIDILITIND.NS ... ✅ Saved 703 rows -> c:\Users\DELL\Documents\PRAXIS\P